In [5]:
import duckdb

In [56]:
con = duckdb.connect(database="dados_duckdb.db", read_only=False)

In [73]:
# update 10002 price
con.execute("update bronze_z0019 set LABST = '99' where NATBR = 10002")

In [74]:
df = con.execute("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
        FROM bronze_z0019
        WHERE data_ingestao >= '2025-10-21'
    )
    WHERE row = 1
    """).fetch_df()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10002,MARTELO,BT10,100,99,z0019_1.csv,2025-10-21 19:34:27.422021,1
1,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-10-21 19:34:27.422021,1
2,10003,CHAVE,BT10,100,50,z0019_1.csv,2025-10-21 19:34:27.422021,1
3,10004,PREGO,BT10,100,70,z0019_2.csv,2025-10-21 19:36:07.245800,1
4,10005,LIMADORA,BT10,100,70,z0019_2.csv,2025-10-21 19:36:07.245800,1


In [75]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao','row'])
df_final = df_final.rename(columns={'NATBR': 'id'})
df_final = df_final.rename(columns={'MAKTX': 'nm_produto'})
df_final = df_final.rename(columns={'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor'})
df_final = df_final.rename(columns={'LABST': 'vl_unitario'})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_unitario
0,10002,MARTELO,BT10,100,99
1,10001,PARAFUSO,BT10,100,100
2,10003,CHAVE,BT10,100,50
3,10004,PREGO,BT10,100,70
4,10005,LIMADORA,BT10,100,70


In [77]:
# enriquecer um pouco mais a tabela
df2 = df_final.copy()
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_unitario
0,10002,MARTELO,BT10,100,99
1,10001,PARAFUSO,BT10,100,100
2,10003,CHAVE,BT10,100,50
3,10004,PREGO,BT10,100,70
4,10005,LIMADORA,BT10,100,70


In [78]:
df2.dtypes


id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_unitario      object
dtype: object

In [82]:
df2 = df2.astype({
    'id': int,
    'nm_produto': str,
    'id_categoria': str,
    'id_fornecedor': int,
    'vl_unitario': float
    })
df2.dtypes



id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_unitario      float64
dtype: object

In [83]:
# criando a table final da camada silver
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id INT,
        nm_produto TEXT,
        id_categoria TEXT,
        id_fornecedor BIGINT,
        vl_unitario FLOAT
    )
""")


In [84]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_unitario
0,10002,MARTELO,BT10,100,99.0
1,10001,PARAFUSO,BT10,100,100.0
2,10003,CHAVE,BT10,100,50.0
3,10004,PREGO,BT10,100,70.0
4,10005,LIMADORA,BT10,100,70.0


In [95]:
df_result = con.execute("SELECT * FROM produtos").fetch_df()
df_result.head(10)

ConnectionException: Connection Error: Connection already closed!

In [2]:
con.execute("""
    INSERT INTO produtos select * from df2
""")

NameError: name 'con' is not defined

In [4]:
con.close()

NameError: name 'con' is not defined